# Temporal IM-Loss SNN Colab Runner

This notebook is the Colab control panel for the temporal SNN experiments. It does not reimplement the model or training loop. It calls the repository scripts so the command-line workflow remains the source of truth.

Main workflow:

1. Load the repo from a Drive zip or Git URL.
2. Install dependencies and verify GPU/PyTorch.
3. Run quick Randman checks.
4. Run the three-condition Randman comparison.
5. Summarize accuracy, losses, spike diagnostics, and plots.
6. Run the output-layer IM lambda sweep.
7. Optionally switch to SHD after placing SHD data in Drive.

Core comparison:

- Non-spiking output with max membrane readout.
- Spiking output counted over time.
- Spiking output counted over time with output-layer IM loss.

## 1. Imports And Helpers

Run this first. The `run(...)` helper executes repository commands and stops the notebook if a command fails.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import textwrap

try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None
from IPython.display import Image, display


def run(args, env=None, cwd=None):
    args = [str(arg) for arg in args]
    print("$", " ".join(args))
    subprocess.run(args, check=True, env=env, cwd=cwd)


def read_csv(path):
    global pd
    if pd is None:
        import pandas as pd
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)


def show_image(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    display(Image(filename=str(path)))


## 2. Repository Setup

For a private repo where you do not own the GitHub permissions, the safest path is a Drive zip. Upload a zip of the repo to Drive, excluding `.venv/`, `runs/`, and large dataset files.

If you already manually unzipped the repo and `%cd`'d into it, this cell will reuse the current folder.

In [ ]:
ZIP_PATH = Path("/content/drive/MyDrive/IM-Loss-Temporal-SNN.zip")
REPO_URL = ""  # Optional. Leave empty for private-repo zip workflow.
REPO_DIR = Path("/content/IM-Loss-Temporal-SNN")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Drive mount skipped or unavailable:", exc)

if Path("train_temporal.py").exists():
    REPO_DIR = Path.cwd()
    print("Using current repository:", REPO_DIR)
elif ZIP_PATH.exists():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    run(["unzip", "-q", ZIP_PATH, "-d", "/content"])
    if not (REPO_DIR / "train_temporal.py").exists():
        candidates = sorted(Path("/content").glob("**/train_temporal.py"))
        if not candidates:
            raise RuntimeError("Could not find train_temporal.py after unzipping.")
        REPO_DIR = candidates[0].parent
    os.chdir(REPO_DIR)
    print("Unzipped repository:", Path.cwd())
elif REPO_URL:
    if not REPO_DIR.exists():
        run(["git", "clone", REPO_URL, REPO_DIR])
    os.chdir(REPO_DIR)
    run(["git", "pull", "--ff-only"])
    print("Cloned repository:", Path.cwd())
else:
    raise RuntimeError(
        "No repo found. Upload ZIP_PATH to Drive, set REPO_URL, or cd into the repo."
    )

required = ["train_temporal.py", "models/temporal_snn.py", "data/randman.py", "scripts/run_temporal_task.py"]
missing = [path for path in required if not Path(path).exists()]
if missing:
    raise RuntimeError(f"Repository is incomplete; missing: {missing}")

print("repo:", Path.cwd())
print("top-level files:")
print("\n".join(sorted(path.name for path in Path.cwd().iterdir())[:30]))


## 3. Install And Runtime Check

This verifies that PyTorch, CUDA, and the CLI entry points are available.

In [ ]:
run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])

import torch

print("python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

run([sys.executable, "train_temporal.py", "--help"])
run([sys.executable, "eval_temporal.py", "--help"])


## 4. Persistent Output Directory And Default Settings

Colab local disk is temporary. Keep checkpoints, CSV files, JSON files, and plots in Drive through `RUNS_ROOT`.

The defaults below target Randman. SHD is configured later in its own section.

In [ ]:
RUNS_ROOT = "/content/drive/MyDrive/im_snn_runs"
Path(RUNS_ROOT).mkdir(parents=True, exist_ok=True)

os.environ.update({
    "RUNS_ROOT": RUNS_ROOT,
    "DATASET": "Randman",
    "DATA_ROOT": "datasets/SHD",
    "EPOCHS": "100",
    "BATCH_SIZE": "64",
    "NUM_WORKERS": "2",
    "HIDDEN_SIZE": "256",
    "NUM_LAYERS": "1",
    "READOUT": "max_membrane",
    "LEARNING_RATE": "0.001",
})

if torch.cuda.is_available():
    os.environ.pop("NO_CUDA", None)
else:
    os.environ["NO_CUDA"] = "1"

print("RUNS_ROOT:", RUNS_ROOT)
print("NO_CUDA:", os.environ.get("NO_CUDA", "0"))
print("key settings:")
for key in ["DATASET", "EPOCHS", "BATCH_SIZE", "NUM_WORKERS", "HIDDEN_SIZE", "READOUT", "LEARNING_RATE"]:
    print(f"  {key}={os.environ.get(key)}")


## 5. Task Runner Helpers

These helpers call `scripts/run_temporal_task.py`. They are useful for dry-runs, partial tests, and full experiment grids.

In [ ]:
def run_task(suite, task_id, dry_run=False):
    cmd = [sys.executable, "scripts/run_temporal_task.py", suite, int(task_id)]
    if dry_run:
        cmd.append("--dry-run")
    run(cmd)


def run_tasks(suite, task_ids, dry_run=False):
    task_ids = list(task_ids)
    print(f"Running {len(task_ids)} task(s) for {suite}: {task_ids}")
    for task_id in task_ids:
        run_task(suite, task_id, dry_run=dry_run)


print("Dry-run examples:")
run_task("ff_output_3way", 0, dry_run=True)
run_task("ff_output_lambda_sweep", 0, dry_run=True)


## 6. Quick Sanity Checks

Run this before expensive experiments. It checks the model stack and does a one-batch Randman train/eval pass.

In [ ]:
run([sys.executable, "scripts/smoke_temporal.py"])
run([
    sys.executable,
    "train_temporal.py",
    "--dataset", "Randman",
    "--output_dir", str(Path(RUNS_ROOT) / "_checks" / "quick_randman"),
    "--epochs", "1",
    "--max_train_batches", "1",
    "--max_eval_batches", "1",
    "--num_workers", "0",
])


## 7. Randman Three-Condition Comparison

This is the primary small benchmark.

Task groups:

- `0-4`: non-spiking max membrane baseline.
- `5-9`: spiking output count.
- `10-14`: spiking output count + rate IM.

Start with one seed using `[0, 5, 10]`. If it works, set `FULL_THREE_WAY=True` to run all 15 tasks.

In [ ]:
FULL_THREE_WAY = False

three_way_tasks = range(15) if FULL_THREE_WAY else [0, 5, 10]
run_tasks("ff_output_3way", three_way_tasks)


## 8. Summarize Randman Three-Condition Runs

This creates CSV, JSON, and a bar plot. Use `--allow_partial` for the one-seed quick run; omit it for the full 5-seed run.

In [ ]:
summary_dir = Path(RUNS_ROOT) / "summary"
summary_dir.mkdir(parents=True, exist_ok=True)

summary_cmd = [
    sys.executable,
    "summarize_ff_output_3way.py",
    "--dataset", "Randman",
    "--run_root", RUNS_ROOT,
    "--output_dir", summary_dir,
]
if not FULL_THREE_WAY:
    summary_cmd.append("--allow_partial")
run(summary_cmd)

randman_summary_csv = summary_dir / "randman_ff_output_3way_summary.csv"
randman_detail_csv = summary_dir / "randman_ff_output_3way_test_eval.csv"
display(read_csv(randman_summary_csv))
display(read_csv(randman_detail_csv).head())
show_image(summary_dir / "randman_ff_output_3way_hist.png")


## 9. Inspect Per-Run Metrics

Each run writes `metrics.csv`. This section finds recent run folders and plots learning curves and output-spike diagnostics from those files.

In [ ]:
def find_run_dirs(root, pattern="metrics.csv"):
    root = Path(root)
    return sorted(
        [path.parent for path in root.rglob(pattern)],
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )


run_dirs = find_run_dirs(Path(RUNS_ROOT) / "randman" / "ff_output_3way")
print(f"found {len(run_dirs)} run directories")
for path in run_dirs[:10]:
    print(path)


In [ ]:
import matplotlib.pyplot as plt


def load_run_frame(run_dir):
    frame = pd.read_csv(Path(run_dir) / "metrics.csv")
    frame["run_dir"] = str(run_dir)
    return frame


def plot_metric_for_runs(run_dirs, metric="val_acc", max_runs=6):
    plt.figure(figsize=(9, 5))
    for run_dir in run_dirs[:max_runs]:
        frame = load_run_frame(run_dir)
        if metric not in frame.columns:
            continue
        label = Path(run_dir).parent.name + "/" + Path(run_dir).name.split("_")[-2]
        plt.plot(frame["epoch"], frame[metric], label=label)
    plt.xlabel("epoch")
    plt.ylabel(metric)
    plt.grid(alpha=0.25)
    plt.legend(fontsize=8)
    plt.show()


plot_metric_for_runs(run_dirs, "val_acc")
plot_metric_for_runs(run_dirs, "val_ce_loss")
plot_metric_for_runs(run_dirs, "val_firing_rate_output")
plot_metric_for_runs(run_dirs, "val_no_output_spike_fraction")


## 10. Output-Layer IM Lambda Sweep

This tests the IM coefficient. It uses spiking output count + output-layer threshold IM by default.

Default grid:

- Lambdas: `0.0, 0.0001, 0.0003, 0.001, 0.003, 0.01`
- Seeds: `2020, 42, 123`

That is 18 tasks. Run after the three-condition comparison looks healthy.

In [ ]:
RUN_LAMBDA_SWEEP = False
os.environ["IM_LAMBDAS"] = "0.0,0.0001,0.0003,0.001,0.003,0.01"
os.environ["SEEDS"] = "2020,42,123"
os.environ["LAMBDA_SWEEP_IM_LOSS_TYPE"] = "threshold"

lambda_count = len([x for x in os.environ["IM_LAMBDAS"].split(",") if x.strip()])
seed_count = len([x for x in os.environ["SEEDS"].split(",") if x.strip()])
lambda_task_count = lambda_count * seed_count
print("lambda sweep tasks:", lambda_task_count)
run_task("ff_output_lambda_sweep", 0, dry_run=True)

if RUN_LAMBDA_SWEEP:
    run_tasks("ff_output_lambda_sweep", range(lambda_task_count))
else:
    print("RUN_LAMBDA_SWEEP is False. Set it to True when ready.")


## 11. Summarize Lambda Sweep

This selects the lambda with the highest mean validation accuracy and creates accuracy/diagnostic plots.

In [ ]:
if RUN_LAMBDA_SWEEP:
    run([
        sys.executable,
        "summarize_lambda_sweep.py",
        "--dataset", "Randman",
        "--loss_type", os.environ["LAMBDA_SWEEP_IM_LOSS_TYPE"],
        "--run_root", RUNS_ROOT,
        "--output_dir", summary_dir,
    ])

    prefix = f"randman_{os.environ['LAMBDA_SWEEP_IM_LOSS_TYPE']}_lambda_sweep"
    display(read_csv(summary_dir / f"{prefix}_summary.csv"))
    display(read_csv(summary_dir / f"{prefix}_detail.csv").head())
    show_image(summary_dir / f"{prefix}_accuracy.png")
    show_image(summary_dir / f"{prefix}_diagnostics.png")
else:
    print("No lambda summary generated because RUN_LAMBDA_SWEEP is False.")


## 12. SHD Data Setup

SHD is the stretch benchmark. Randman needs no data files because samples are generated by code. SHD is a real dataset and must be stored outside Git.

Expected Drive files:

```text
/content/drive/MyDrive/datasets/SHD/shd_train.h5
/content/drive/MyDrive/datasets/SHD/shd_test.h5
```

If they are missing, the next cell can download the compressed SHD files and decompress them. Skip SSC for this project unless you explicitly extend the repo later.

In [ ]:
SHD_ROOT = Path("/content/drive/MyDrive/datasets/SHD")
SHD_ROOT.mkdir(parents=True, exist_ok=True)

DOWNLOAD_SHD = False
if DOWNLOAD_SHD:
    run(["wget", "-nc", "https://compneuro.net/datasets/shd_train.h5.gz", "-P", SHD_ROOT])
    run(["wget", "-nc", "https://compneuro.net/datasets/shd_test.h5.gz", "-P", SHD_ROOT])
    for name in ["shd_train.h5", "shd_test.h5"]:
        if not (SHD_ROOT / name).exists():
            run(["gunzip", "-k", str(SHD_ROOT / f"{name}.gz")])

print("SHD root:", SHD_ROOT)
print("train exists:", (SHD_ROOT / "shd_train.h5").exists())
print("test exists:", (SHD_ROOT / "shd_test.h5").exists())
print("files:", [path.name for path in sorted(SHD_ROOT.glob("shd_*"))])


## 13. SHD Smoke Test And Optional Runs

Only run this after `shd_train.h5` and `shd_test.h5` exist. Start with one batch. Then run one seed. Then consider the full SHD grid if Colab runtime allows.

In [ ]:
RUN_SHD_ONE_BATCH = False

if RUN_SHD_ONE_BATCH:
    if not (SHD_ROOT / "shd_train.h5").exists() or not (SHD_ROOT / "shd_test.h5").exists():
        raise FileNotFoundError("SHD files are missing. Run the SHD download/setup cell first.")

    run([
        sys.executable,
        "train_temporal.py",
        "--dataset", "SHD",
        "--data_root", SHD_ROOT,
        "--output_dir", str(Path(RUNS_ROOT) / "_checks" / "quick_shd"),
        "--epochs", "1",
        "--max_train_batches", "1",
        "--max_eval_batches", "1",
        "--num_workers", "0",
    ])
else:
    print("RUN_SHD_ONE_BATCH is False. Enable it after SHD files are ready.")


In [ ]:
RUN_SHD_THREE_WAY = False
FULL_SHD_THREE_WAY = False

if RUN_SHD_THREE_WAY:
    os.environ.update({
        "DATASET": "SHD",
        "DATA_ROOT": str(SHD_ROOT),
        "EPOCHS": "100",
        "BATCH_SIZE": "64",
        "NUM_WORKERS": "2",
    })
    shd_tasks = range(15) if FULL_SHD_THREE_WAY else [0, 5, 10]
    run_tasks("ff_output_3way", shd_tasks)
else:
    print("RUN_SHD_THREE_WAY is False. Randman should be completed first.")


## 14. Summarize SHD Three-Condition Runs

Use this after SHD three-way runs finish. For quick one-seed SHD checks, allow partial summaries. For final results, use full multi-seed summaries.

In [ ]:
RUN_SHD_SUMMARY = False

if RUN_SHD_SUMMARY:
    shd_summary_cmd = [
        sys.executable,
        "summarize_ff_output_3way.py",
        "--dataset", "SHD",
        "--run_root", RUNS_ROOT,
        "--output_dir", summary_dir,
    ]
    if not FULL_SHD_THREE_WAY:
        shd_summary_cmd.append("--allow_partial")
    run(shd_summary_cmd)
    display(read_csv(summary_dir / "shd_ff_output_3way_summary.csv"))
    show_image(summary_dir / "shd_ff_output_3way_hist.png")
else:
    print("RUN_SHD_SUMMARY is False.")


## 15. Result Files To Keep

Main result files are under `RUNS_ROOT` in Drive.

Per run:

- `config.json`: full run configuration.
- `metrics.csv`: per-epoch train/validation curves and diagnostics.
- `best.pth`: best validation checkpoint.
- `last.pth`: final checkpoint.
- `metrics.json`: final test metrics from the best checkpoint.

Summary folder:

- `*_summary.csv`: aggregated accuracy table.
- `*_test_eval.csv` or `*_detail.csv`: per-seed details.
- `*.png`: plots for report figures.

For final reporting, prefer multi-seed mean and standard deviation, not one-seed results.

In [ ]:
print("RUNS_ROOT contents:")
for path in sorted(Path(RUNS_ROOT).glob("*")):
    print(path)

print("\nSummary contents:")
if summary_dir.exists():
    for path in sorted(summary_dir.glob("*")):
        print(path)
